In [26]:
import pandas as pd

# Load dataset (assuming a CSV file named 'data.csv')
df = pd.read_csv('houses_dataset.csv')

# Extract the 'description' column
descriptions = df['basic_info_description']

In [52]:
# There are a few common pitfalls when using asyncio with sync SDKs (like google-genai), especially when running at scale.
# If your code gets "stuck" after 16 tasks, here are almost always the reasons:
#   (1) Python's default event loop policy uses `SelectorEventLoop` (on Unix), which (on some systems) defaults to 1024 fds,
#       but *aiohttp* or *requests* or even aiofiles have platform limits. 
#   (2) The Gemini model SDK here is fully synchronous, and you use `asyncio.to_thread` for every call. This creates real threads.
#       The Python threadpool size is usually very small (default 5!) unless you override it.
#       So, launching 100s of "async" tasks doesn't help: only 5 will actually execute at a time. If one blocks, others queue up.
#   (3) Your system may have a ulimit (file descriptors), or your network (NAT) may hit ephemeral port exhaustion, but this is unlikely with 16.
#   (4) Jupyter and notebooks (esp. with `nest_asyncio` or in some environments) handle signals and event loops weirdly under the hood.

# The *actual cause* in your system is the use of asyncio's default threadpool (ThreadPoolExecutor), which is only 5 threads,
# so only 5 requests actually run concurrently (the rest await their turn). If your provider (the Gemini API) is slow, you'll hang.

# TL;DR: Since the Google SDK is fully synchronous & you use to_thread, you MUST bump up the default threadpool size to match your concurrency.

# Here is the rewritten code with comments to fix this problem:

import base64
import os
import asyncio
from google import genai
from google.genai import types

import dotenv
dotenv.load_dotenv()

from tqdm import tqdm
import concurrent.futures

def build_analysis_prompt(text):
    return f"Encode this description: {text}"

def get_generate_content_config():
    tools = [
        types.Tool(googleSearch=types.GoogleSearch())
    ]
    return types.GenerateContentConfig(
        thinkingConfig={"thinkingBudget": 0},
        media_resolution="MEDIA_RESOLUTION_LOW",
        tools=tools,
        system_instruction=[
            types.Part.from_text(text="You are a Real Estate Quantitative Analyst. Your task is to analyze property descriptions and convert qualitative language into strict numerical values for a regression machine learning model.\n\nConstraints:\n1. Be Objective: Ignore marketing fluff (\"breathtaking,\" \"stunning\") unless it correlates with material value.\n2. Be Cynical: Assume \"cozy\" means small, and \"TLC\" means expensive repairs.\n3. Output Format: You must output valid JSON only. No markdown formatting, no conversational text.\n\nScoring Rubric (Strict Adherence Required):\n\n1. luxury_score (Integer 1-10)\nReflects the quality of finishes and materials.\n* 1-3 (Low): Laminate counters, linoleum/carpet, builder-grade cabinets, \"potential.\"\n* 4-6 (Mid): Standard stainless steel, some hardwood, \"well maintained,\" \"clean.\"\n* 7-8 (High): Granite/Quartz, hardwood throughout, crown molding, \"gourmet kitchen.\"\n* 9-10 (Premium): Sub-Zero/Viking appliances, imported stone, smart home, custom millwork, \"architectural masterpiece.\"\n\n2. renovation_index (Integer 0-3)\nReflects the recency and extent of updates.\n* 0 (Vintage/Gut): \"TLC,\" \"Handyman special,\" \"Needs work,\" \"Original condition.\"\n* 1 (Dated): Liveable but outdated style (e.g., 1990s oak cabinets, older tile).\n* 2 (Updated): \"New appliances,\" \"Updated bath,\" \"New flooring\" (Partial updates).\n* 3 (Turn-Key): \"Fully remodeled,\" \"Gut renovation,\" \"New Construction,\" \"Everything new.\"\n\n3. distress_probability (Float 0.0 - 1.0)\nThe likelihood the seller is desperate or the asset is distressed.\n* 0.0: Standard listing.\n* 0.3: \"Relocating,\" \"Must sell.\"\n* 0.7: \"As-is,\" \"Cash only,\" \"Investor special.\"\n* 1.0: \"Foreclosure,\" \"Bank owned,\" \"Short sale,\" \"Court ordered.\"\n\n4. marketing_hype_ratio (Float 0.0 - 1.0)\nThe ratio of subjective adjectives to objective facts.\n* 0.1: Dry, factual description (e.g., \"3 bed, 2 bath, new roof\").\n* 0.9: Pure fluff (e.g., \"Feel the magic of this breathtaking sanctuary...\").\n\n5. location_premium_inferred (Integer 0-2)\nDoes the text mention specific high-value location keywords?\n* 0: No specific location highlights.\n* 1: \"Corner lot,\" \"Cul-de-sac,\" \"Quiet street.\"\n* 2: \"Waterfront,\" \"Ocean view,\" \"Golf course,\" \"City skyline view.\"")
        ],
    )

async def async_generate_encoded_description(client, model, description, semaphore):
    async with semaphore:
        contents = [
            types.Content(
                role="user",
                parts=[
                    types.Part.from_text(
                        text=build_analysis_prompt(description)
                    ),
                ],
            ),
        ]
        config = get_generate_content_config()
        result_text = ""
        try:
            def run():
                output = ""
                for chunk in client.models.generate_content_stream(
                    model=model,
                    contents=contents,
                    config=config,
                ):
                    output += chunk.text
                return output
            # FIX: asyncio.to_thread uses the default ThreadPoolExecutor, which is tiny. We'll set the executor at the event loop level.
            loop = asyncio.get_running_loop()
            result_text = await loop.run_in_executor(None, run)
        except Exception as e:
            err_msg = str(e)
            if "rate limit" in err_msg.lower() or "quota" in err_msg.lower() or "429" in err_msg:
                print(f"[RATE LIMIT DETECTED] {err_msg}")
            else:
                print(f"[API ERROR] {err_msg}")
            result_text = f'{{"error": "{err_msg}"}}'
        return result_text

async def async_generate_all(descriptions, max_concurrency=100):
    client = genai.Client(
        api_key=os.getenv("GOOGLE_API_KEY"),
    )
    model = "gemini-flash-lite-latest"
    semaphore = asyncio.Semaphore(max_concurrency)

    results = [None] * len(descriptions)
    pbar = tqdm(total=len(descriptions), desc="Processing", unit="desc")

    async def wrapped_task(i, desc):
        res = await async_generate_encoded_description(client, model, desc, semaphore)
        if isinstance(res, str) and ("rate limit" in res.lower() or "quota" in res.lower() or "429" in res):
            print(f"RATE LIMITED on index {i}: {res}")
        results[i] = res
        pbar.update(1)

    # FIX: Use a custom ThreadPoolExecutor that matches or slightly exceeds max_concurrency
    # This ensures that asyncio.to_thread/run_in_executor can launch parallel blocking calls
    loop = asyncio.get_running_loop()
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=max_concurrency + 4)
    loop.set_default_executor(executor)

    tasks = [
        asyncio.create_task(wrapped_task(i, desc))
        for i, desc in enumerate(descriptions)
    ]
    await asyncio.gather(*tasks)
    pbar.close()
    # Clean up the executor explicitly (helps in Jupyter)
    executor.shutdown(wait=True)
    return results

import nest_asyncio
nest_asyncio.apply()

if not isinstance(descriptions, list):
    descriptions_list = list(descriptions)
else:
    descriptions_list = descriptions

print("Launching Gemini analysis (rate limit status will be printed below if encountered)...")

# IMPORTANT: Run in a context that allows for many threads (ThreadPoolExecutor set above). This prevents being "stuck" at a small number of parallel requests.
all_results = asyncio.run(async_generate_all(descriptions_list, max_concurrency=10))

# Now all_results is a list of JSON strings with the model's outputs for each description.



Launching Gemini analysis (rate limit status will be printed below if encountered)...


Processing: 100%|██████████| 4853/4853 [06:25<00:00, 12.59desc/s]


In [53]:
import pandas as pd

# Parse all_results (list of JSON strings or dicts) into a DataFrame
parsed_results = []
for r in all_results:
    if isinstance(r, str):
        try:
            parsed = eval(r)
        except Exception as e:
            print(f"Could not parse row: {r}\nError: {e}")
            parsed = {}
        parsed_results.append(parsed)
    else:
        parsed_results.append(r)

df = pd.json_normalize(parsed_results)

# Save the DataFrame to CSV for later use
df.to_csv("gemini_encoded_descriptions.csv", index=False)
print("Saved parsed model outputs to gemini_encoded_descriptions.csv")


Saved parsed model outputs to gemini_encoded_descriptions.csv
